In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval
except Exception:
    pass

In [2]:
from collections import Counter
from collections.abc import Iterable, Mapping

import numpy as np

from dataeval import Ontology
from dataeval.core import label_alignment
from dataeval.data import Conform, Relabel, merge_datasets
from dataeval.protocols import DatasetMetadata

In [3]:
class ToyDataset:
    """A minimal image-classification dataset: one-hot targets + an index2label."""

    def __init__(self, dataset_id: str, labels: Iterable[int], index2label: Mapping[int, str]) -> None:
        self._labels = list(labels)
        self._index2label = dict(index2label)
        self.metadata = DatasetMetadata(id=dataset_id, index2label=self._index2label)

    def __len__(self) -> int:
        return len(self._labels)

    def __getitem__(self, index: int):
        onehot = np.zeros(len(self._index2label), dtype=np.float32)
        onehot[self._labels[index]] = 1.0
        return np.zeros((3, 8, 8), dtype=np.float32), onehot, {"id": index}


def labels_from_counts(counts: dict[str, int], index2label: dict[int, str]) -> list[int]:
    """Expand ``{class_name: count}`` into a per-image list of label indices."""
    name_to_index = {name: index for index, name in index2label.items()}
    return [name_to_index[name] for name, count in counts.items() for _ in range(count)]

In [4]:
reference_ontology = Ontology.from_hierarchy({
    "aircraft": {"airliner": None, "fighter jet": None},
    "land vehicle": {"sedan": None, "pickup truck": None},
    "watercraft": {"frigate": None, "cargo ship": None},
})
print(reference_ontology)
print("reference vocabulary:", {i: reference_ontology.concept(c).label for i, c in enumerate(reference_ontology.ids)})

Ontology(9 concepts, 3 roots, 6 leaves, 0 external)
reference vocabulary: {0: 'aircraft', 1: 'airliner', 2: 'fighter jet', 3: 'land vehicle', 4: 'sedan', 5: 'pickup truck', 6: 'watercraft', 7: 'frigate', 8: 'cargo ship'}


In [5]:
REFERENCE_VOCAB = {0: "sedan", 1: "pickup truck", 2: "frigate", 3: "cargo ship"}
reference_raw = ToyDataset(
    "reference",
    labels_from_counts({"sedan": 6, "pickup truck": 4, "frigate": 5, "cargo ship": 3}, REFERENCE_VOCAB),
    REFERENCE_VOCAB,
)

reference_alignment = label_alignment(reference_raw.metadata.get("index2label", {}).values(), reference_ontology)
reference = Conform(reference_raw, [Relabel(reference_alignment["class_remap"], reference_ontology)])

print("reference images:", len(reference))
print("reference now uses the vocabulary:", reference.metadata.get("index2label"))

reference images: 18
reference now uses the vocabulary: {0: 'aircraft', 1: 'airliner', 2: 'fighter jet', 3: 'land vehicle', 4: 'sedan', 5: 'pickup truck', 6: 'watercraft', 7: 'frigate', 8: 'cargo ship'}


In [6]:
INCOMING_VOCAB = {0: "submarine", 1: "frigate", 2: "sedan", 3: "fighter jet"}
incoming = ToyDataset(
    "incoming",
    labels_from_counts({"submarine": 4, "frigate": 3, "sedan": 5, "fighter jet": 2}, INCOMING_VOCAB),
    INCOMING_VOCAB,
)
print("incoming vocabulary:", incoming.metadata.get("index2label"))

incoming vocabulary: {0: 'submarine', 1: 'frigate', 2: 'sedan', 3: 'fighter jet'}


In [7]:
incoming_alignment = label_alignment(incoming.metadata.get("index2label", {}).values(), reference_ontology)
for c in incoming_alignment["correspondences"]:
    print(f"  {c.source:>12}  {c.relation:<11} -> {c.target}")
print("out-of-vocabulary:", incoming_alignment["unaligned_source"])

       frigate  equivalent  -> frigate
         sedan  equivalent  -> sedan
   fighter jet  equivalent  -> fighter jet
out-of-vocabulary: ('submarine',)


In [8]:
relabel = Relabel(incoming_alignment["class_remap"], reference_ontology)
incoming_conformed = Conform(incoming, [relabel])

print("kept", len(incoming_conformed), "of", len(incoming), "images")
print("dropped (out-of-vocabulary):", dict(relabel.dropped))
print(
    "incoming now uses the reference vocabulary:",
    incoming_conformed.metadata.get("index2label") == reference.metadata.get("index2label"),
)

kept 10 of 14 images
dropped (out-of-vocabulary): {0: 'submarine'}
incoming now uses the reference vocabulary: True


In [9]:
merged = merge_datasets(reference, incoming_conformed)
print("merged images:", len(merged))

index2label = merged.metadata.get("index2label", {})
counts = Counter(index2label[int(np.argmax(datum[1]))] for datum in merged)
print("per-class counts:", dict(counts))

merged images: 28
per-class counts: {'sedan': 11, 'pickup truck': 4, 'frigate': 8, 'cargo ship': 3, 'fighter jet': 2}


In [10]:
try:
    merge_datasets(reference, incoming)
except ValueError as error:
    print("without conforming:", str(error).splitlines()[0])

without conforming: merge_datasets requires all datasets to share the same 'index2label'. Conform them to a common vocabulary first (see dataeval.data.Conform / Relabel).
